# Tome 13, Chapter 7: The circle method II, three primes and Waring's problem (Python companion)

Exercises first, solutions at the end. We count representations as sums of three primes and of two primes with the fast Fourier transform, compare them with the singular series, test Weyl's inequality and Hua's lemma on cubes, and find the numbers that need many cubes. The computer science application: **exact convolution with the number-theoretic transform**, the integer twin of the FFT that the counts in this chapter call for. Requires `numpy`.

In [ ]:
import math, time
import numpy as np

def prime_sieve(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return s

def conv_power(a, s):
    # s-fold self-convolution of a real array by the FFT (float)
    L = 1 << int(np.ceil(np.log2(s * len(a))))
    return np.fft.irfft(np.fft.rfft(a, L) ** s, L)[: s * (len(a) - 1) + 1]

def singular3(N, primes):
    v = 1.0
    for p in primes:
        v *= (1 - 1 / (p - 1) ** 2) if N % p == 0 else (1 + 1 / (p - 1) ** 3)
    return v

## Exercises

**P7.1 (★) Three primes.** For odd $N\le 2\cdot10^5$, compute the number $r_3(N)$ of ordered triples of primes with sum $N$, and $R(N)=\sum\Lambda\Lambda\Lambda$. Compare $R(N)$ with $\frac12\mathfrak S(N)N^2$ and $r_3(N)$ with $\frac{\mathfrak S(N)N^2}{2\log^3N}$ (Exercise 7.3). Which odd $N\ge7$ has the fewest representations relative to the prediction?

**P7.2 (★★) Two primes.** For even $n\le10^6$, compute $r_2(n)$, the number of ordered pairs of primes with sum $n$. Check Goldbach's conjecture in this range. Compare $R_2(n)=\sum\Lambda\Lambda$ with Hardy and Littlewood's conjectured main term $\mathfrak S_2(n)n$, $\mathfrak S_2(n)=2C_2\prod_{p\mid n,p>2}\frac{p-1}{p-2}$. How large is the relative error, typically, at $n\approx10^6$?

**P7.3 (★★) Weyl and Hua for cubes.** (a) For $k=3$ and $P=2000$, compute $|f(\alpha)|$ for $\alpha=\frac1q+\frac1{4qP^3}$ with several $q$, and compare with $P\frac{|S(q,1)|}q$ (Lemma 7.9) and with $P(\frac1q+\frac1P+\frac q{P^3})^{1/4}$ (Theorem 7.7 without its constant and its $P^\varepsilon$). (b) Compute $\int_0^1|f|^8=\sum_nr_{4,3}(n)^2$ exactly for $P=50,100,200,400$ and estimate the exponent; Hua's lemma predicts $5+\varepsilon$.

**P7.4 (★) Waring for cubes.** For each $n\le10^6$, find the least number of positive cubes with sum $n$. List the $n$ that need $9$ and $8$, and the largest $n$ in range that need $7$ and $6$.

**P7.5 (★★) Exact convolution (computer science).** The FFT computes $r_3(N)$ in floating point and we round. (a) Estimate the rounding error of the float FFT in P7.1 by comparing with an exact computation. (b) Implement the number-theoretic transform modulo the prime $p=998244353=119\cdot2^{23}+1$, which has a primitive $2^{23}$-th root of unity, and use it with two primes and the Chinese remainder theorem to compute $r_3(N)$ exactly for $N\le2\cdot10^5$. Compare speed and results with the float FFT.

## Solutions

### P7.1

In [ ]:
M = 200000
isp = prime_sieve(M); primes = np.nonzero(isp)[0]
lam = np.zeros(M + 1)
for p in primes:
    pk = p
    while pk <= M:
        lam[pk] = math.log(p); pk *= p
R = conv_power(lam, 3)
r3 = np.rint(conv_power(isp.astype(float), 3)).astype(np.int64)
small = primes[primes < 5000]
rows = []
for N in [1001, 10001, 100001, 199999]:
    S = singular3(N, small)
    rows.append((N, r3[N], R[N] / (0.5 * S * N * N), r3[N] / (S * N * N / (2 * math.log(N) ** 3))))
for N, c, a, b in rows:
    print(f"N = {N:6d}: r3 = {c:10d},  R/(S N^2/2) = {a:.4f},  r3/(S N^2/(2 log^3 N)) = {b:.3f}")
Ns = np.arange(1001, M + 1, 2)
ratio = np.array([R[N] / (0.5 * singular3(N, small[:60]) * N * N) for N in Ns[::50]])
print("R/(S N^2/2) over odd N in [1001, 2e5] (sampled):  min %.4f  max %.4f" % (ratio.min(), ratio.max()))
print("smallest r3 for odd N >= 7:", [(N, r3[N]) for N in range(7, 40, 2)])

**What it shows.** The weighted count $R(N)$ follows $\frac12\mathfrak S(N)N^2$ closely: the ratio is $0.945$ at $N=1001$, $0.993$ at $N=10001$ and $0.9994$ at $N=10^5$ and $2\cdot10^5$. The unweighted count $r_3(N)$ is $1.95$ times the naive prediction $\frac{\mathfrak S N^2}{2\log^3N}$ at $N=1001$ and still $1.50$ times it at $2\cdot10^5$, because $\log p<\log N$ for most primes $p<N$: the asymptotic of Exercise 7.3 is true but slow, and $\frac{\mathfrak S(N)}2\sum_{a+b+c=N}\frac1{\log a\log b\log c}$ is a much better approximation. Small odd numbers already have several representations ($7=2+2+3$ in three orders), which is why Helfgott's theorem was believable long before it was proved.

### P7.2

In [ ]:
M2 = 10 ** 6
isp2 = prime_sieve(M2)
r2 = np.rint(conv_power(isp2.astype(float), 2)).astype(np.int64)
evens = np.arange(4, M2 + 1, 2)
print("every even n in [4, 10^6] is a sum of two primes:", bool((r2[evens] > 0).all()))
print("even n with exactly one or two ordered representations:", [n for n in evens if r2[n] <= 2][:12])
lam2 = np.zeros(M2 + 1)
for p in np.nonzero(isp2)[0]:
    pk = p
    while pk <= M2:
        lam2[pk] = math.log(p); pk *= p
R2 = conv_power(lam2, 2)
C2 = 0.6601618158468696
odd_small = np.nonzero(isp2[:1001])[0][1:]
def S2(n):
    # 2 C_2 prod_{p | n, p > 2} (p-1)/(p-2); trial division up to sqrt(10^6)
    v = 2 * C2; m = n
    while m % 2 == 0: m //= 2
    for p in odd_small:
        if m % p == 0:
            v *= (p - 1) / (p - 2)
            while m % p == 0: m //= p
    if m > 1:
        v *= (m - 1) / (m - 2)
    return v
errs = np.array([R2[n] / (S2(n) * n) - 1 for n in range(999000, 1000001, 2)])
print(f"n in [999000, 10^6]: relative error of S2(n) n: mean {errs.mean():+.4f}, standard deviation {errs.std():.4f}")

**What it shows.** Every even number from $4$ to $10^6$ is a sum of two primes; only $4=2+2$, $6=3+3$, $8=3+5$ and $12=5+7$ have one or two ordered representations. (The record verification, by Oliveira e Silva, Herzog and Pardi, reaches $4\cdot10^{18}$.) The Hardy–Littlewood prediction $\mathfrak S_2(n)n$ for $R_2(n)$ is accurate to about $0.6\%$ (one standard deviation) near $10^6$, with no visible bias (mean relative error $-0.04\%$): the conjectured asymptotic is as good as Vinogradov's proved one for three primes, but nobody can control the minor arcs.

### P7.3

In [ ]:
P = 2000
x = np.arange(1, P + 1, dtype=np.float64)
x3 = x ** 3
def f_at(a, q, beta):
    xi = np.arange(1, P + 1, dtype=np.int64)
    frac = (a * (xi ** 3 % q) % q) / q + (beta * x3) % 1.0
    return abs(np.exp(2j * np.pi * frac).sum())
for q in [1, 2, 7, 31, 101, 1009, 10007, 100003, 1000003]:
    val = f_at(1, q, 1 / (4 * q * P ** 3))
    bound = P * (1 / q + 1 / P + q / P ** 3) ** 0.25
    print(f"q = {q:8d}: |f| = {val:9.1f},   P(1/q+1/P+q/P^3)^(1/4) = {bound:8.1f}")
print()
for P4 in [50, 100, 200, 400]:
    cubes = np.zeros(P4 ** 3 + 1); cubes[np.arange(1, P4 + 1) ** 3] = 1
    r4 = np.rint(conv_power(cubes, 4))
    moment = float((r4 ** 2).sum())
    print(f"P = {P4:3d}: integral of |f|^8 = {moment:.4e},  log / log P = {math.log(moment) / math.log(P4):.3f}")

**What it shows.** (a) For $q\le P$ the sum is close to $P\frac{|S(q,1)|}q$, as Lemma 7.9 predicts: $1811$ at $q=1$ (the small $\beta$ costs $10\%$), $1352$ at $q=7$, $661$ at $q=31$. For primes $q\equiv2\pmod3$ with $q\le P$ ($q=2$ and $101$), cubing permutes the residues, so $S(q,1)=0$ and $|f|$ is tiny. At $q=7$ the true value even exceeds the bound without its constant: the cubes modulo $7$ take only the values $0,\pm1$, so $S(7,1)$ is large. Once $q>P$ the sum is incomplete and behaves like a random walk of $P$ steps, of size about $\sqrt P\approx45$, while Weyl's bound stays above $300$: it is an inequality for the worst case, not a prediction. (b) The eighth moment grows like $P^5$: the measured exponent $\frac{\log\int|f|^8}{\log P}$ is $5.066$, $5.028$, $5.008$, $4.999$ for $P=50,100,200,400$, consistent with Hua's $P^{5+\varepsilon}$; the true order is believed to be a constant times $P^5$.

### P7.4
Dynamic programming over the number of cubes: $c(n)=1+\min_{x^3\le n}c(n-x^3)$, vectorized over $n$.

In [ ]:
M4 = 10 ** 6
INF = 99
cubes = [x ** 3 for x in range(1, int(round(M4 ** (1 / 3))) + 2) if x ** 3 <= M4]
c = np.full(M4 + 1, INF, dtype=np.int16); c[0] = 0
# after round j, c[n] is the least number of cubes summing to n if it is at most j
for _ in range(9):
    new = c.copy()
    for t in cubes:
        np.minimum(new[t:], c[:-t] + 1, out=new[t:])
    c = new
print("need 9 cubes:", np.nonzero(c == 9)[0].tolist())
print("need 8 cubes:", np.nonzero(c == 8)[0].tolist())
for k in [7, 6, 5]:
    idx = np.nonzero(c == k)[0]
    print(f"need {k} cubes: {len(idx)} numbers, the largest {idx.max()}")

**What it shows.** Only $23$ and $239$ need nine cubes ($23=2\cdot8+7\cdot1$), as Wieferich and Kempner proved for all integers. Fifteen numbers need eight, the largest being $454$; $121$ numbers need seven, the largest being $8042$, and it is conjectured that $8042$ is the last one, so that $G(3)$ would be at most $6$. Below $10^6$, $3919$ numbers need six, the largest $864455$. Linnik proved in 1943 that every large number is a sum of seven cubes; $G(3)\ge4$ because cubes are $0,\pm1$ modulo $9$. Corollary 7.13 is far weaker for cubes (it needs $s\ge12$ for the singular series), which shows how much the special methods for small $k$ add.

### P7.5
(a) The exact counts are integers; the float FFT returns them with an error that we can measure as the distance to the nearest integer. (b) The NTT is the discrete Fourier transform over $\mathbb Z/p\mathbb Z$, with $3$ a primitive root modulo $998244353$; it is exact, and two primes plus the Chinese remainder theorem give integers up to about $10^{18}$.

In [ ]:
def ntt(a, p, g, invert=False):
    a = a.copy(); n = len(a)
    j = 0
    for i in range(1, n):            # bit reversal
        bit = n >> 1
        while j & bit:
            j ^= bit; bit >>= 1
        j ^= bit
        if i < j: a[i], a[j] = a[j], a[i]
    length = 2
    while length <= n:
        w = pow(g, (p - 1) // length, p)
        if invert: w = pow(w, p - 2, p)
        ws = np.array([pow(w, k, p) for k in range(length // 2)], dtype=object)
        a = a.reshape(-1, length)
        u = a[:, : length // 2].copy(); v = (a[:, length // 2:] * ws) % p
        a = np.concatenate([(u + v) % p, (u - v) % p], axis=1).reshape(-1)
        length <<= 1
    if invert:
        ninv = pow(n, p - 2, p); a = (a * ninv) % p
    return a

def conv_power_exact(ind, s, primes=((998244353, 3), (469762049, 3))):
    L = 1 << int(np.ceil(np.log2(s * len(ind))))
    res = []
    for p, g in primes:
        a = np.zeros(L, dtype=object); a[: len(ind)] = ind.astype(object)
        A = ntt(a, p, g)
        res.append(ntt((A ** s) % p, p, g, invert=True))
    (p1, _), (p2, _) = primes
    m1inv = pow(p1, -1, p2)
    x1, x2 = res
    return [int(u) + p1 * ((int(v) - int(u)) * m1inv % p2) for u, v in zip(x1[: s * (len(ind) - 1) + 1], x2[: s * (len(ind) - 1) + 1])]

Msmall = 20000
ind = prime_sieve(Msmall).astype(np.int64)
t0 = time.time(); exact = conv_power_exact(ind, 3); t1 = time.time()
flt = conv_power(ind.astype(float), 3); t2 = time.time()
err = max(abs(flt[n] - exact[n]) for n in range(len(exact)))
print(f"N <= {Msmall}: NTT {t1 - t0:.2f} s, float FFT {t2 - t1:.4f} s, largest float error {err:.2e}, largest count {max(exact)}")
print("rounded float FFT equals the exact counts:", all(int(round(flt[n])) == exact[n] for n in range(len(exact))))
frac = np.abs(r3 - conv_power(isp.astype(float), 3)[: len(r3)])
print(f"N <= {M}: largest distance of the float FFT from an integer: {frac.max():.2e}")

**What it shows.** In double precision, the float FFT returns the counts of this chapter within $3\cdot10^{-8}$ of the integers, far below $\frac12$, so rounding is safe. But the error grows with the size of the counts and the length of the transform, roughly like $\varepsilon_{\text{mach}}\cdot\max|r|\cdot\log L$; for sums of many powers, or longer ranges, it eventually exceeds $\frac12$ and the rounded answer is wrong without warning. The NTT never errs. Our pure Python version is about $300$ times slower than numpy's FFT; compiled NTTs, as used in competitive programming libraries and in computer algebra (for instance for multiplying huge integers, as in the Schönhage–Strassen algorithm), are within a small factor of the float FFT.

*References.* J. M. Pollard, "The fast Fourier transform in a finite field", *Math. Comp.* 25 (1971), 365–374. A. Schönhage and V. Strassen, "Schnelle Multiplikation großer Zahlen", *Computing* 7 (1971), 281–292. T. Oliveira e Silva, S. Herzog and S. Pardi, "Empirical verification of the even Goldbach conjecture and computation of prime gaps up to $4\cdot10^{18}$", *Math. Comp.* 83 (2014), 2033–2060.